# HyDE：先生成假设答案再检索

HyDE（Hypothetical Document Embeddings）先让生成模型写一段**假设答案**，再把这段模型输出送进向量检索。它改变的是检索表达，不是把假设答案当成事实。

本页使用《南瓜书》中的两个问题：先用原问题检索，再由 `glm-4-flash` 生成一段假设答案，最后用同一个本地 BGE 向量模型重新检索。参考答案和正确页码只在两次检索都完成后用于核对，不会传给生成模型或检索器。

重新执行需要 `.env` 中的 `ZHIPUAI_API_KEY`，并会产生真实 API 调用。当前实验只观察这两道题，不能把结果外推到其他资料。


## 原理、适用条件与代码要点

HyDE 由 Gao 等人在 2023 年的 [Precise Zero-Shot Dense Retrieval without Relevance Labels](https://arxiv.org/abs/2212.10496) 中提出。它包含三步：先让 LLM 在没有外部资料的情况下写一段假设的、像知识库文章一样的回答；再把假设文字（也可以和原问题一起）编码成向量；然后在真实文档库中找最接近的文档。假设文字不需要真实存在，作用是把问题的意图改写到更接近文档的语义空间。

![HyDE 原理](./figures/HyDE.png)

它的优点是不用相关性标注就能工作，生成的文字通常比短问题包含更多检索线索，适合用户用口语提问、而资料使用专业术语的情况。缺点是检索前多了一次模型调用，而且模型可能写错。这里让模型只写章节摘要和概念名，不写公式或结论；最终回答仍要依据检索到的原文。

本页后面的代码已经用 `glm-4-flash` 和随教程提供的向量库完成正式实验；这里的片段只用来说明调用顺序，不另存一份结果。

```python
from common.nontraining_utils import llm_call

question = '电影哪吒之魔童降世讲述了什么故事？'
hypothetical = llm_call(
    '写一到两句可能出现在知识库目录或章节摘要中的文字，只列主题和概念名。\n'
    f'问题：{question}'
)
retrieved_docs = search(question + '\n' + hypothetical, top_k=8)
```

参考：Gao 等人的论文与 LangChain 的 hypothetical document embeddings（假设文档向量）示例。

In [1]:
import json
import sys
from pathlib import Path

def find_tutorial_root(start: Path) -> Path:
    candidates = [start, *start.parents, start / "notebook" / "C7 高级 RAG 技巧"]
    for folder in candidates:
        if (folder / "data" / "dataset/manifest.json").is_file() and (folder / "common" / "eval_utils.py").is_file():
            return folder
    raise FileNotFoundError("找不到教程数据目录，请从教程所在目录运行")

TUTORIAL_ROOT = find_tutorial_root(Path.cwd().resolve())
sys.path.insert(0, str(TUTORIAL_ROOT))
from common.eval_utils import emit_tutorial_audit
from common.nontraining_utils import (
    RAG_LLM_MODEL, build_reused_chunk_search, load_annotation,
    load_query_only, llm_call, rank_and_coverage,
)

CASE_IDS = ["hyde_gini_impurity", "hyde_cv_reliability"]
queries = load_query_only(CASE_IDS)
search = build_reused_chunk_search()

records = []
for item in queries:
    question = item["query"]
    before = search(question, top_k=8)
    hypothetical = llm_call(
        "你是 HyDE 检索器。只写一到两句可能出现在教材目录或章节摘要中的文字。"
        "写出问题描述对应的标准方法名或指标名，并列出相关概念名；不要解释这些概念。"
        "禁止写公式、数值、大小方向、因果结论、操作步骤和页码；禁止声称看过资料。\n"
        f"用户问题：{question}",
        max_tokens=260,
    )
    after = search(question + "\n" + hypothetical, top_k=8)
    records.append({
        "case_id": item["id"],
        "query": question,
        "before": before,
        "after": after,
        "model_outputs": {"hypothetical_document": hypothetical},
    })

def emit_standard(record, role):
    annotation = record["annotation"]
    before = rank_and_coverage(record["before"], annotation["expected_pages"])
    after = rank_and_coverage(record["after"], annotation["expected_pages"])
    compact = lambda metrics: {key: metrics[key] for key in ("pages", "first_required_rank", "required_page_coverage")}
    evidence_pages = sorted({
        int(span.get("page"))
        for claim in annotation.get("essential_evidence_spans", [])
        if isinstance(claim, dict)
        for span in claim.get("spans", [])
        if isinstance(span, dict) and str(span.get("page", "")).isdigit()
    })
    payload = {
        "case_id": record["case_id"],
        "method": "HyDE",
        "role": role,
        "query": record["query"],
        "before": compact(before),
        "after": compact(after),
        "model_outputs": record.get("model_outputs", {}),
        "annotation_check_after_result": {
            "expected_pages": annotation.get("expected_pages", []),
            "evidence_pages": evidence_pages,
        },
    }
    emit_tutorial_audit(payload)
    return compact(before), compact(after)

# 只有两次检索和假设答案都完成后，才读取评估标注。
for record in records:
    record["annotation"] = load_annotation(record["case_id"])
    role = "main" if record["case_id"] == "hyde_gini_impurity" else "check"
    before, after = emit_standard(record, role)
    print(f"\n--- HyDE 对照：{record['query']}（模型={RAG_LLM_MODEL}）---")
    print("问题：", record["query"])
    for label, metrics in (("改前", before), ("改后", after)):
        rank = metrics["first_required_rank"] if metrics["first_required_rank"] is not None else "未命中"
        coverage = f"{metrics['required_page_coverage']:.0%}"
        pages = "、".join(str(page) for page in metrics["pages"])
        print(f"{label}：首个必要页排名 {rank}；必要页覆盖率 {coverage}；返回页面 {pages}")
    print("真实模型生成的假设文字：", record["model_outputs"]["hypothetical_document"])



--- HyDE 对照：决策树怎样用类别概率的平方和来表示一个节点的不纯度？（模型=glm-4-flash）---
问题： 决策树怎样用类别概率的平方和来表示一个节点的不纯度？
改前：首个必要页排名 2；必要页覆盖率 100%；返回页面 45、49、50、97、51、101、100、77
改后：首个必要页排名 1；必要页覆盖率 100%；返回页面 49、45、50、108、100、101、51、97
真实模型生成的假设文字： 标准方法名：基尼指数（Gini Index）
相关概念名：类别概率、不纯度

--- HyDE 对照：怎样让算法比较不被一次训练集和测试集的偶然性左右？（模型=glm-4-flash）---
问题： 怎样让算法比较不被一次训练集和测试集的偶然性左右？
改前：首个必要页排名 1；必要页覆盖率 100%；返回页面 19、18、16、101、23、15、44、28
改后：首个必要页排名 1；必要页覆盖率 100%；返回页面 19、18、16、101、14、15、151、28
真实模型生成的假设文字： 标准方法名：交叉验证
相关概念名：训练集、测试集、模型泛化能力


## 结果解读

主要题的第 49 页排名从 2 提到 1；复查题的覆盖保持不变。保存的假设文字只补充了‘基尼指数’等检索词，没有生成公式或结论。它只用于查找资料，不能直接当作答案。

## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[判断是否需要继续检索](判断是否需要继续检索.ipynb)

